# QUERY 2
Nombre de banco, cuenta de origen y monto de la max. transacción USD de cada banco.

In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────
USE_SAMPLE = True          # False:usa los datasets completos
CANTIDAD_FILAS = 5000       # solo se usa si USE_SAMPLE = True

RUTA_DE_DATASETS             = "../../datasets"
NOMBRE_ARCHIVO_TRANSACCIONES = "LI-Small_Trans.csv"
NOMBRE_ARCHIVO_CUENTAS       = "LI-Small_accounts.csv"
RUTA_TRANSACCIONES_SAMPLE    = f"{RUTA_DE_DATASETS}/transacciones_sample.csv"
RUTA_CUENTAS_SAMPLE          = f"{RUTA_DE_DATASETS}/accounts_sample.csv"
RUTA_RESULTADO_QUERY2        = "query2_result.csv"

In [2]:
import pandas as pd

def normalizar_bank_id(serie):
    return serie.str.strip().str.lstrip("0").replace("", "0")

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)
cuentas_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_CUENTAS}",
    dtype={"Bank ID": "string", "Account Number": "string"}
)
columnas_cuentas_originales = cuentas_completas.columns.tolist()

In [3]:
if USE_SAMPLE:
    transacciones = transacciones_completas.sample(n=CANTIDAD_FILAS, random_state=42).copy()

    transacciones["From Bank Normalized"] = normalizar_bank_id(transacciones["From Bank"])
    transacciones["To Bank Normalized"]   = normalizar_bank_id(transacciones["To Bank"])
    cuentas_completas["Bank ID Normalized"] = normalizar_bank_id(cuentas_completas["Bank ID"])

    cuentas_relevantes = pd.concat([
        transacciones[["From Bank Normalized", "Account"]].rename(
            columns={"From Bank Normalized": "Bank ID Normalized", "Account": "Account Number"}
        ),
        transacciones[["To Bank Normalized", "Account.1"]].rename(
            columns={"To Bank Normalized": "Bank ID Normalized", "Account.1": "Account Number"}
        )
    ], ignore_index=True).dropna().drop_duplicates()

    cuentas = cuentas_completas.merge(
        cuentas_relevantes, on=["Bank ID Normalized", "Account Number"], how="inner"
    )[columnas_cuentas_originales]

    guardar_csv(transacciones.drop(columns=["From Bank Normalized", "To Bank Normalized"]), RUTA_TRANSACCIONES_SAMPLE)
    guardar_csv(cuentas, RUTA_CUENTAS_SAMPLE)
    print(f"Modo SAMPLE: {len(transacciones)} transacciones, {len(cuentas)} cuentas")
    print(f"  → guardado en {RUTA_TRANSACCIONES_SAMPLE}")
    print(f"  → guardado en {RUTA_CUENTAS_SAMPLE}")
else:
    transacciones = transacciones_completas.copy()
    cuentas       = cuentas_completas.copy()
    print(f"Modo COMPLETO: {len(transacciones)} transacciones, {len(cuentas)} cuentas")

transacciones.head()

Modo SAMPLE: 100000 transacciones, 130524 cuentas
  → guardado en ../../datasets/transacciones_sample.csv
  → guardado en ../../datasets/accounts_sample.csv


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From Bank Normalized,To Bank Normalized
5681437,2022/09/08 21:03,0014,811143E30,0146477,8112133A0,6160.23,Mexican Peso,6160.23,Mexican Peso,Cheque,0,14,146477
1082292,2022/09/01 14:11,026922,80330D020,027356,81A02AFC0,16946.26,Yuan,16946.26,Yuan,Credit Card,0,26922,27356
6752037,2022/09/10 08:15,009735,8042FDA10,019931,804EB7BC0,1772.13,Yuan,1772.13,Yuan,Cash,0,9735,19931
4834527,2022/09/07 14:16,019836,80642DBB0,0217073,8064D5D00,73361.71,Yen,73361.71,Yen,Cheque,0,19836,217073
1867807,2022/09/02 07:43,024633,801EFE020,0017256,81C11F940,179.87,US Dollar,179.87,US Dollar,Cheque,0,24633,17256


In [4]:
transacciones_usd = transacciones[
    transacciones["Payment Currency"] == "US Dollar"
].copy()
transacciones_usd["From Bank Normalized"] = normalizar_bank_id(transacciones_usd["From Bank"])
transacciones_usd["Amount Paid"] = pd.to_numeric(transacciones_usd["Amount Paid"], errors="coerce")
transacciones_usd = transacciones_usd.dropna(subset=["Amount Paid"])

bancos = cuentas[["Bank ID", "Bank Name"]].copy()
bancos["From Bank Normalized"] = normalizar_bank_id(bancos["Bank ID"])
bancos = bancos[["From Bank Normalized", "Bank ID", "Bank Name"]].drop_duplicates(subset=["From Bank Normalized"])

transacciones_con_banco = transacciones_usd.merge(bancos, on="From Bank Normalized", how="inner")

idx_maximos = transacciones_con_banco.groupby("From Bank Normalized")["Amount Paid"].idxmax()
resultado_query2 = transacciones_con_banco.loc[
    idx_maximos, ["Bank ID", "Bank Name", "Account", "Amount Paid"]
].rename(columns={"Amount Paid": "Max Amount"}).sort_values(by="Bank ID").reset_index(drop=True)

guardar_csv(resultado_query2, RUTA_RESULTADO_QUERY2)
resultado_query2.head()

,Bank ID,Bank Name,Account,Max Amount
0,0,Italy Bank #18,8003B0E40,1.392827e+04
1,1,First Bank of Portland,80121C170,4.174583e+08
2,10073,China Bank #114,8042DA970,3.608000e+01
3,10154,China Bank #448,80445C950,5.578667e+04
4,10311,China Bank #14,807B63AD0,1.453781e+05
